### Setup Data loader

In [1]:
%load_ext autoreload
%autoreload 2

from notebooks.create_deepGP import ServiceGP, prepare_chained_data, DynamicServiceChain
import gpytorch
from agent.components.commons import ServiceFeatureMapping, ServiceType
from typing import List
import torch

from notebooks.create_deepGP import get_prepared_metrics_df

torch.set_default_dtype(torch.float64)


In [2]:


converted_df = get_prepared_metrics_df()

QR_MAP = ServiceFeatureMapping(ServiceType.QR, [0, 1])
CV_MAP = ServiceFeatureMapping(ServiceType.CV, [2, 3, 4])
PC_MAP = ServiceFeatureMapping(ServiceType.PC, [5, 6])

repetitions = 3
chain_definition = ([QR_MAP] * repetitions) + ([CV_MAP] * repetitions) + ([PC_MAP] * repetitions)


train_loader, test_x, test_y, scaler_X = prepare_chained_data(converted_df, chain_definition, test_size=0.2)

### Setup the Structure


In [4]:

model = DynamicServiceChain(chain_definition, num_inducing=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Dynamic MLL setup
mlls = [
    gpytorch.mlls.VariationalELBO(model.likelihoods[i], model.gp_layers[i], num_data=len(train_loader.dataset))
    for i in range(len(chain_definition))
]

# Training Loop
model.train()
for epoch in range(500):
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        # Returns a tuple of distributions: (qr_dist, cv_dist, pc_dist)
        distributions = model(x_batch)

        # Calculate sum of losses dynamically
        loss = sum([-mlls[i](distributions[i], y_batch[:, i]) for i in range(len(mlls))])

        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")
    if epoch % 50 == 0:
        for i, likelihood in enumerate(model.likelihoods):
            print(f"Noise: {likelihood.noise.item():.4f}")

/home/boris/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 0 | Loss: 11.7971
Noise: 0.6982
Noise: 0.6982
Noise: 0.6982
Noise: 0.6982
Noise: 0.6980
Noise: 0.6981
Noise: 0.6980
Noise: 0.6976
Noise: 0.6977
Epoch 10 | Loss: 9.3940
Epoch 20 | Loss: 8.0256
Epoch 30 | Loss: 7.1415
Epoch 40 | Loss: 6.6190
Epoch 50 | Loss: 5.9053
Noise: 0.4927
Noise: 0.4694
Noise: 0.4708
Noise: 0.4684
Noise: 0.4446
Noise: 0.4446
Noise: 0.4460
Noise: 0.4458
Noise: 0.4454
Epoch 60 | Loss: 5.5049
Epoch 70 | Loss: 5.0877
Epoch 80 | Loss: 4.5686
Epoch 90 | Loss: 4.1149
Epoch 100 | Loss: 3.7324
Noise: 0.3019
Noise: 0.2825
Noise: 0.2844
Noise: 0.2732
Noise: 0.2626
Noise: 0.2594
Noise: 0.2691
Noise: 0.2726
Noise: 0.2722
Epoch 110 | Loss: 3.3401
Epoch 120 | Loss: 2.8755


KeyboardInterrupt: 

In [ ]:
model.eval()
all_samples = []
num_mc = 100  # Number of Monte Carlo simulations per data point
num_services = len(model.configs)

with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-3):
    for _ in range(num_mc):
        # 1. model(test_x) now returns a tuple of N distributions
        distributions = model(test_x)

        # 2. Sample from each distribution in the chain
        # We use a list comprehension to handle all N services dynamically
        samples = [dist.sample() for dist in distributions]

        # 3. Stack along the last dimension to keep (N_test, N_services)
        s = torch.stack(samples, dim=-1)
        all_samples.append(s)

    # 4. Stack all MC trials into a single NumPy array
    # Shape: (num_mc, num_test_samples, num_services)
    combined_samples = torch.stack(all_samples, dim=0).cpu().numpy()

print(f"Inference complete. Samples shape: {combined_samples.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculate the mean and std for EACH test case across simulations
# combined_samples shape: (num_mc, num_test_samples, num_services)
means_per_config = combined_samples.mean(axis=0)  # Shape: (N_test, N_services)
stds_per_config = combined_samples.std(axis=0)    # Shape: (N_test, N_services)

# 2. Average across all test configurations to get the global chain profile
global_means = means_per_config.mean(axis=0)      # Shape: (N_services,)
global_stds = stds_per_config.mean(axis=0)        # Shape: (N_services,)

# Extract service names for the X-axis labels
stages = range(len(chain_definition))

plt.figure(figsize=(12, 6))

# 3. Plot the Global Staircase
# fmt='-s' connects the dots to show the throughput "decay" or bottlenecking
plt.errorbar(stages, global_means, yerr=global_stds, fmt='-s', capsize=10,
             color='navy', ecolor='red', elinewidth=3, markersize=8,
             label='Average Chain Performance')

# 4. Annotate the Average Sigma for each stage
# for i, stage_name in enumerate(stages):
#     plt.text(i + 0.1, global_means[i], f'σ={global_stds[i]:.3f}',
#              va='bottom', fontweight='bold', color='red', fontsize=9)

plt.title(f"Global Throughput Bottleneck ({len(stages)} Stages)")
plt.ylabel("Average Scaled Throughput (0-1)")
plt.xlabel("Service Pipeline Sequence")

plt.grid(axis='y', alpha=0.3)
plt.legend(loc='upper right')

# 5. Shaded area showing Hardware Diversity (Best vs Worst config)
plt.fill_between(stages,
                 means_per_config.min(axis=0),
                 means_per_config.max(axis=0),
                 color='gray', alpha=0.1, label='Hardware Spread')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# --- MODEL VALIDATION ---
# 1. Get the mean prediction for each service across Monte Carlo samples
# combined_samples shape: (num_mc, N_test, num_services)
predicted_means = combined_samples.mean(axis=0)

# 2. Extract the ground truth (test_y)
actual_y = test_y.cpu().numpy()

# 3. Calculate Error for each service dynamically
# We iterate based on the number of services defined in the model
for i, config in enumerate(model.configs):
    # Calculate RMSE for the i-th service in the chain
    rmse = np.sqrt(np.mean((predicted_means[:, i] - actual_y[:, i]) ** 2))
    print(f"{config.service_type.value} Service RMSE: {rmse:.4f}")